In [1]:
import torch
from torch import nn

In [2]:
x = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
target = 2 * x + 1

In [9]:
model = nn.Linear(1, 1)
with torch.no_grad():
    model.weight.fill_(0.0)
    model.bias.fill_(0.0)

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.1, momentum=0.0, weight_decay=0.0
)
print("初始w/b:", model.weight.item(), model.bias.item())
print("学习率:", optimizer.param_groups[0]["lr"])

初始w/b: 0.0 0.0
学习率: 0.1


In [25]:
optimizer.zero_grad(set_to_none=True)
w_before = model.weight.item()
b_before = model.bias.item()

In [26]:
prediction = model(x)
loss = criterion(prediction, target)
print("预测:", prediction.detach())
print("loss:", loss.item())
print("backward前梯度:", model.weight.grad, model.bias.grad)

预测: tensor([[-2.6480],
        [-1.0800],
        [ 0.4880],
        [ 2.0560],
        [ 3.6240]])
loss: 0.6353920698165894
backward前梯度: None None


In [27]:
loss.backward()

grad_w = model.weight.grad.item()
grad_b = model.bias.grad.item()
print("backward后梯度:", grad_w, grad_b)
print("backward后w/b:", model.weight.item(), model.bias.item())
print("原来的w/b:", w_before, b_before)

backward后梯度: -1.7280001640319824 -1.0240000486373901
backward后w/b: 1.5679999589920044 0.4880000054836273
原来的w/b: 1.5679999589920044 0.4880000054836273


In [28]:
lr = optimizer.param_groups[0]["lr"]

# 仅计算普通数字，还没有修改模型
expected_w = w_before - lr * grad_w
expected_b = b_before - lr * grad_b

optimizer.step()

actual_w = model.weight.item()
actual_b = model.bias.item()
print("公式预测w/b:", expected_w, expected_b)
print("step实际w/b:", actual_w, actual_b)
print("核对w:", abs(expected_w - actual_w) < 1e-6)
print("核对b:", abs(expected_b - actual_b) < 1e-6)
print("step后学习率:", optimizer.param_groups[0]["lr"])

with torch.no_grad():
    loss_after = criterion(model(x), target).item()
print("更新前/后loss:", loss.item(), loss_after)

公式预测w/b: 1.7407999753952026 0.5904000103473663
step实际w/b: 1.7408000230789185 0.590399980545044
核对w: True
核对b: True
step后学习率: 0.1
更新前/后loss: 0.6353920698165894 0.302141398191452


In [8]:
print("清除前w/b:", model.weight.item(), model.bias.item())
print("清除前grad:", model.weight.grad, model.bias.grad)

optimizer.zero_grad(set_to_none=True)

print("清除后w/b:", model.weight.item(), model.bias.item())
print("清除后grad:", model.weight.grad, model.bias.grad)

清除前w/b: 0.4000000059604645 0.10000000149011612
清除前grad: tensor([[-8.]]) tensor([-2.])
清除后w/b: 0.4000000059604645 0.10000000149011612
清除后grad: None None
